In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession 
    .builder 
    .appName("Streaming from Kafka") 
    .config("spark.streaming.stopGracefullyOnShutdown", True) 
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0')
    .config("spark.sql.shuffle.partitions", 4)
    .master("local[*]") 
    .getOrCreate()
)

spark

In [28]:
from pyspark.sql.functions import input_file_name, split, element_at,col
df=spark.read.csv(r'apple_products_202512161628.csv')
df_with_path = df.withColumn("full_path", input_file_name())
df_new=df_with_path.withColumn("file_name",element_at(split(element_at(split("full_path",'/'),-1),'_'),-1))
df_new2=df_new.withColumn("file_timestamp",element_at(split("file_name",'_'),-1))
df_new2.drop("full_path").show()

+--------------------+--------------------+-----+----------+------+-------------------+-----------------+-----------------+----------------+-----------+----+----------------+----------------+
|                 _c0|                 _c1|  _c2|       _c3|   _c4|                _c5|              _c6|              _c7|             _c8|        _c9|_c10|       file_name|  file_timestamp|
+--------------------+--------------------+-----+----------+------+-------------------+-----------------+-----------------+----------------+-----------+----+----------------+----------------+
|        Product Name|         Product URL|Brand|Sale Price|   Mrp|Discount Percentage|Number Of Ratings|Number Of Reviews|             Upc|Star Rating| Ram|202512161628.csv|202512161628.csv|
|APPLE iPhone 8 Pl...|https://www.flipk...|Apple|     49900| 49900|                  0|             3431|              356|MOBEXRGV7EHHTGUH|        4.6|2 GB|202512161628.csv|202512161628.csv|
|APPLE iPhone 8 Pl...|https://www.flipk.